# DDP profiling exploration

Quick walk-through of `torch.profiler` output from a short DDP run.
I usually run a 5-step trace, then load it in TensorBoard with the profiler plugin.

Steps:
1. Run `python -m src.train_single --config configs/default.yaml` with the profiler context wrapped around the inner loop.
2. Open the trace JSON in TensorBoard.
3. Look at the kernel breakdown to find the slow ops.

In [ ]:
import os
import torch
from src.profile_util import profile_steps
from src.model import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = build_model('resnet18', num_classes=10).to(device)
x = torch.randn(8, 3, 224, 224, device=device)
y = torch.randint(0, 10, (8,), device=device)
crit = torch.nn.CrossEntropyLoss()
opt = torch.optim.SGD(m.parameters(), lr=0.01)

with profile_steps(out_dir='./profiles', wait=1, warmup=1, active=3) as prof:
    for _ in range(6):
        opt.zero_grad()
        loss = crit(m(x), y)
        loss.backward()
        opt.step()
        prof.step()

print('trace written to ./profiles')

## What I usually see

- `aten::convolution_backward` is the largest single bucket on resnet50 -- close to half the wall time.
- NCCL all-reduce shows up as `ncclKernel_AllReduce_*` and overlaps with the next layer's backward when the bucket size is right.
- DataLoader workers usually disappear from the trace because they're separate processes; if your epoch is data-bound, you'll see big gaps in the GPU timeline.